# Imports

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ecdf

## load data

In [ ]:
raw_df = pd.read_csv('results.csv', index_col=0)
df = raw_df.copy()

# plotting definitions

In [ ]:
labels={'Variant': 'Viral Variant', 
        'tversky_index': 'PLIF Recall', 
        # 'ByTotalInteractions': 'Total Number of Interactions',
        'ByTotalInteractions': 'Total num interactions',
        'ByEverything': 'Atomic Level', 
        'ByInteractionType': 'Interaction Type', 
        'ByInteractionTypeAndAtomTypes': 'Interaction Type and Atom Type',
        # 'ByInteractionTypeAndResidueTypeAndBBorSC': 'Interaction Type and Residue Type and Backbone vs Sidechain',
        'ByInteractionTypeAndResidueTypeAndBBorSC': 'By interaction Type, residue type, and backbone vs sidechain',
        # 'ByInteractionTypeAndResidueTypeAndNumber': 'Interaction Type and Residue Type and Number',
        'ByInteractionTypeAndResidueTypeAndNumber': 'By interaction type and residue type and num',
        'Fingerprint Specificity': 'Fingerprint Specificity',
        'SARS2': 'SARS-CoV-2',
        'MERS': 'MERS-CoV',}
# fingerprint_specificity_order = ['Interaction Type and Residue Type and Number', 'Total Number of Interactions']
fingerprint_specificity_order = ['By interaction Type, residue type, and backbone vs sidechain', 'Total num interactions']
variant_order = ['SARS-CoV-2', 'MERS-CoV']
variant_colors = ["#2077b5", "#0b8140"]
sns.set_style("white")

In [ ]:
for key, value in labels.items():
    df = df.replace(key, value)

In [ ]:
plot_df = df[(df['Docked_Variant'] == df['Crystal_Variant'])&(df["provenance"].isin(fingerprint_specificity_order))]

In [ ]:
# Calculate ECDF data for each combination
def calculate_ecdf(data):
    calculated_ecdf = ecdf(data).cdf
    x = np.insert(calculated_ecdf.quantiles, 0, 0)
    y = np.insert(calculated_ecdf.probabilities, 0, 0)
    return x, y

# Create empty lists to store the results
x_values = []
y_values = []
variants = []
specs = []

# Calculate ECDF for each combination
for variant in plot_df['Docked_Variant'].unique():
    for spec in plot_df['provenance'].unique():
        mask = (plot_df['Docked_Variant'] == variant) & (plot_df['provenance'] == spec)
        x, y = calculate_ecdf(plot_df[mask]['tversky_index'])
        
        # Store the results
        x_values.extend(x)
        y_values.extend(y)
        variants.extend([variant] * len(x))
        specs.extend([spec] * len(x))

# Create a DataFrame with the ECDF data
ecdf_df = pd.DataFrame({
    'tversky_index': x_values,
    'cumulative_probability': y_values,
    'percent': [y * 100 for y in y_values],  # Convert to percentage
    'Docked_Variant': variants,
    'Fingerprint Specificity': specs
})

In [ ]:
# Create the plot
plt.figure(figsize=(8, 6))
g = sns.lineplot(data=ecdf_df, 
                 x='tversky_index', 
                 y='percent',
                 hue='Docked_Variant',
                 hue_order=variant_order,
                 style='Fingerprint Specificity',
                 style_order=fingerprint_specificity_order,
                 errorbar=None,
                 estimator=None,
                 palette=variant_colors,
                 drawstyle='steps-post',
                 )

plt.xlabel('PLIF Recall')
plt.ylabel("Fraction of Structures (%)")

# Remove the legend border
plt.legend(frameon=False)

handles, labels = g.get_legend_handles_labels()
for t in g.legend_.get_texts():
    if t.get_text() in ["Docked_Variant", "Fingerprint Specificity"]:
        t.set_text('')
    if t.get_text() == 'Interaction Type and Residue Type and Number':
        t.set_text('By interaction type and residue type and num')
    if t.get_text() == 'Total Number of Interactions':
        t.set_text('Total interactions')
plt.tight_layout()
plt.savefig('to_self.pdf', bbox_inches='tight')

In [ ]:
plot_df = df[(df['Docked_Variant'] != df['Crystal_Variant'])&(df["provenance"].isin(fingerprint_specificity_order))]

In [ ]:
df[df['provenance'] == 'Atomic Level'].groupby(['Docked_Variant', 'Crystal_Variant']).count()

In [ ]:
# Calculate ECDF data for each combination
def calculate_ecdf(data):
    calculated_ecdf = ecdf(data).cdf
    x = np.insert(calculated_ecdf.quantiles, 0, 0)
    y = np.insert(calculated_ecdf.probabilities, 0, 0)
    return x, y

# Create empty lists to store the results
x_values = []
y_values = []
variants = []
specs = []

# Calculate ECDF for each combination
for variant in plot_df['Docked_Variant'].unique():
    for spec in plot_df['provenance'].unique():
        mask = (plot_df['Docked_Variant'] == variant) & (plot_df['provenance'] == spec)
        x, y = calculate_ecdf(plot_df[mask]['tversky_index'])
        
        # Store the results
        x_values.extend(x)
        y_values.extend(y)
        variants.extend([variant] * len(x))
        specs.extend([spec] * len(x))

# Create a DataFrame with the ECDF data
ecdf_df = pd.DataFrame({
    'tversky_index': x_values,
    'cumulative_probability': y_values,
    'percent': [y * 100 for y in y_values],  # Convert to percentage
    'Docked_Variant': variants,
    'Fingerprint Specificity': specs
})

In [ ]:
# Create the plot
plt.figure(figsize=(8, 6))
g = sns.lineplot(data=ecdf_df, 
                 x='tversky_index', 
                 y='percent',
                 hue='Docked_Variant',
                 hue_order=variant_order,
                 style='Fingerprint Specificity',
                 style_order=fingerprint_specificity_order,
                 errorbar=None,
                 estimator=None,
                 palette=variant_colors,
                 drawstyle='steps-post',
                 )

plt.xlabel('PLIF Recall')
plt.ylabel("Fraction of Structures (%)")

# Remove the legend border
plt.legend(frameon=False)

handles, labels = g.get_legend_handles_labels()
for t in g.legend_.get_texts():
    if t.get_text() in ["Docked_Variant", "Fingerprint Specificity"]:
        t.set_text('')
    if t.get_text() == 'Interaction Type and Residue Type and Number':
        t.set_text('By interaction type and residue type and num')
    if t.get_text() == 'Total Number of Interactions':
        t.set_text('Total interactions')
plt.tight_layout()
plt.savefig('to_cross_crystal_structure.pdf', bbox_inches='tight')